In [1]:
!pip install sentence-transformers scikit-learn pandas numpy tqdm

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [2]:
from pathlib import Path
from itertools import chain
from collections import Counter, defaultdict
import re
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer

In [3]:
DATA_DIR = Path("./data/ebm_nlp_2_00")
DOCS_DIR = DATA_DIR / "documents"

assert DATA_DIR.exists(), f"DATA_DIR does not exist: {DATA_DIR}"
assert DOCS_DIR.exists(), f"DOCS_DIR does not exist: {DOCS_DIR}"
print("Data directory is ready：", DATA_DIR)

Data directory is ready： data\ebm_nlp_2_00


In [ ]:
#Find which documents belong to a given category (P/I/O) in the train or test set from the corresponding annotation directory of the dataset.
def get_doc_ids(split="train", label_type="participants"):
    """
    label_type: 'participants', 'interventions', 'outcomes'
    split: 'train' or 'test'
    """
    if split == "test":
        split = "test/gold"#Location of the ground truth for the test set

    ann_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
    )

    if not ann_dir.exists():
        print("Directory does not exist：", ann_dir)
        return []

    doc_ids = [p.name.split(".")[0] for p in ann_dir.glob("*.AGGREGATED.ann")]
    return sorted(doc_ids)

In [ ]:
def load_labels_for_doc(doc_id, label_type="participants", split="train"):#read labels
    if split == "test":
        split = "test/gold"

    ann_path = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
        / f"{doc_id}.AGGREGATED.ann"
    )

    if not ann_path.exists():
        return None

    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f if line.strip() != ""]
    return labels

In [ ]:
def hierarchical_to_bio(tags):#Convert the original hierarchical labels to BIO format
    bio = []
    prev = 0

    for t in tags:
        t = int(t)
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")
        prev = t

    return bio


def convert_all_labels_to_bio(label_lists):
    return [hierarchical_to_bio(doc_labels) for doc_labels in label_lists]

In [ ]:
def find_document_file(doc_id):
    """
    Only read the .tokens files, because they correspond one-to-one with the token-level labels.
    """
    token_file = DOCS_DIR / f"{doc_id}.tokens"
    if token_file.exists() and token_file.is_file():
        return token_file
    return None


def load_document_tokens_for_doc(doc_id):
    """
    Read the .tokens content of a document.
    Prefer to process it as “one token per line”.
    """
    doc_path = find_document_file(doc_id)
    if doc_path is None:
        print(f"[WARN] Not found {doc_id}.tokens")
        return None

    with open(doc_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip() != ""]

    return lines


def load_documents(doc_ids): #Read all documents corresponding to a batch of document IDs
    docs = []
    kept_ids = []
    for doc_id in tqdm(doc_ids):
        tokens = load_document_tokens_for_doc(doc_id)
        if tokens is not None:
            docs.append(tokens)
            kept_ids.append(doc_id)
    return kept_ids, docs

In [ ]:
# Load the P / I / O data for the given document IDs
doc_ids_p = get_doc_ids("train", "participants")
doc_ids_i = get_doc_ids("train", "interventions")
doc_ids_o = get_doc_ids("train", "outcomes")

test_doc_ids_p = get_doc_ids("test", "participants")
test_doc_ids_i = get_doc_ids("test", "interventions")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print("train P/I/O:", len(doc_ids_p), len(doc_ids_i), len(doc_ids_o))
print("test  P/I/O:", len(test_doc_ids_p), len(test_doc_ids_i), len(test_doc_ids_o))

train P/I/O: 4609 4746 4681
test  P/I/O: 189 187 190


In [10]:
common_train_ids = sorted(set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o))
common_test_ids = sorted(set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o))

print("Number of shared training documents:", len(common_train_ids))
print("Number of shared test documents:", len(common_test_ids))

Number of shared training documents: 4457
Number of shared test documents: 184


In [ ]:
common_train_ids, train_docs = load_documents(common_train_ids)#Read the tokens of the shared documents
common_test_ids, test_docs = load_documents(common_test_ids)

print("Number of successfully loaded training documents:", len(common_train_ids))
print("Number of successfully loaded test documents:", len(common_test_ids))

100%|██████████| 184/184 [00:01<00:00, 95.04it/s] 

Number of successfully loaded training documents: 4457
Number of successfully loaded test documents: 184


In [ ]:
def load_label_bundle(doc_ids, split="train"):#Read the P / I / O labels of the shared documents
    p_labels = []
    i_labels = []
    o_labels = []
    kept_ids = []

    for doc_id in tqdm(doc_ids):
        lp = load_labels_for_doc(doc_id, "participants", split)
        li = load_labels_for_doc(doc_id, "interventions", split)
        lo = load_labels_for_doc(doc_id, "outcomes", split)

        if lp is None or li is None or lo is None:
            continue

        p_labels.append(lp)
        i_labels.append(li)
        o_labels.append(lo)
        kept_ids.append(doc_id)

    return kept_ids, p_labels, i_labels, o_labels


common_train_ids_labels, p_train_raw, i_train_raw, o_train_raw = load_label_bundle(common_train_ids, "train")
common_test_ids_labels, p_test_raw, i_test_raw, o_test_raw = load_label_bundle(common_test_ids, "test")

print(len(common_train_ids_labels), len(common_test_ids_labels))

100%|██████████| 184/184 [00:05<00:00, 32.98it/s]

4457 184


In [ ]:
#Ensure that doc_ids and docs are aligned
train_doc_map = {doc_id: tokens for doc_id, tokens in zip(common_train_ids, train_docs)}
test_doc_map = {doc_id: tokens for doc_id, tokens in zip(common_test_ids, test_docs)}

train_ids_final = [doc_id for doc_id in common_train_ids_labels if doc_id in train_doc_map]
test_ids_final = [doc_id for doc_id in common_test_ids_labels if doc_id in test_doc_map]

train_docs_final = [train_doc_map[x] for x in train_ids_final]
test_docs_final = [test_doc_map[x] for x in test_ids_final]

train_idx_map = {doc_id: idx for idx, doc_id in enumerate(common_train_ids_labels)}
test_idx_map = {doc_id: idx for idx, doc_id in enumerate(common_test_ids_labels)}

p_train = [p_train_raw[train_idx_map[x]] for x in train_ids_final]
i_train = [i_train_raw[train_idx_map[x]] for x in train_ids_final]
o_train = [o_train_raw[train_idx_map[x]] for x in train_ids_final]

p_test = [p_test_raw[test_idx_map[x]] for x in test_ids_final]
i_test = [i_test_raw[test_idx_map[x]] for x in test_ids_final]
o_test = [o_test_raw[test_idx_map[x]] for x in test_ids_final]

print("Final training set:", len(train_ids_final))
print("Final test set :", len(test_ids_final))

Final training set: 4457
Final test set : 184


In [ ]:
p_train_bio = convert_all_labels_to_bio(p_train)#Convert to BIO
i_train_bio = convert_all_labels_to_bio(i_train)
o_train_bio = convert_all_labels_to_bio(o_train)

p_test_bio = convert_all_labels_to_bio(p_test)
i_test_bio = convert_all_labels_to_bio(i_test)
o_test_bio = convert_all_labels_to_bio(o_test)

In [15]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [16]:
from nltk.tokenize import sent_tokenize

def build_text_and_token_offsets(tokens):
    """
    Join the token list into a complete text and record the start and end character positions of each token.
    """
    pieces = []
    token_offsets = []
    current_pos = 0

    for i, tok in enumerate(tokens):
        start = current_pos
        pieces.append(tok)
        current_pos += len(tok)
        end = current_pos

        token_offsets.append((start, end))

        if i < len(tokens) - 1:
            pieces.append(" ")
            current_pos += 1

    text = "".join(pieces)
    return text, token_offsets


def get_sentence_char_spans(text, sentences):
    """
    Find the corresponding character span in the full text for each sentence segmented by NLTK.
    """
    spans = []
    search_start = 0

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        start = text.find(sent, search_start)
        if start == -1:
            continue

        end = start + len(sent)
        spans.append((start, end))
        search_start = end

    return spans


def char_span_to_token_span(token_offsets, sent_start, sent_end):
    """
    Map the sentence's character span to a token span.
    Return: (start_token_idx, end_token_idx), left-closed and right-open.
    """
    token_start = None
    token_end = None

    for i, (tok_start, tok_end) in enumerate(token_offsets):
        if tok_end > sent_start and tok_start < sent_end:
            if token_start is None:
                token_start = i
            token_end = i + 1

    if token_start is None:
        return None

    return (token_start, token_end)


def split_tokens_into_sentences(tokens):
    """
    Use NLTK to split the text into sentences first, then align them back to token spans.
    Return:
    sentences: List[List[str]]
    spans: List[(start_idx, end_idx)]
    """
    if not tokens:
        return [], []

    full_text, token_offsets = build_text_and_token_offsets(tokens)
    sentence_texts = sent_tokenize(full_text)
    sent_char_spans = get_sentence_char_spans(full_text, sentence_texts)

    sentences = []
    spans = []

    for sent_text, (char_start, char_end) in zip(sentence_texts, sent_char_spans):
        token_span = char_span_to_token_span(token_offsets, char_start, char_end)
        if token_span is None:
            continue

        start_idx, end_idx = token_span
        sent_tokens = tokens[start_idx:end_idx]

        sentences.append(sent_tokens)
        spans.append((start_idx, end_idx))

    return sentences, spans

In [ ]:
def sentence_has_entity(bio_tags):#Convert BIO labels from the token level to the sentence level, and determine which of P, I, or O the whole sentence belongs to.
    return any(x in {"B", "I"} for x in bio_tags)

def get_sentence_label(p_sent, i_sent, o_sent):
    """
    Assign a main label to each sentence:
    P / I / O / NONE.
    If a sentence contains multiple entities, choose the one with the highest count.
    """
    p_count = sum(x in {"B", "I"} for x in p_sent)
    i_count = sum(x in {"B", "I"} for x in i_sent)
    o_count = sum(x in {"B", "I"} for x in o_sent)

    counts = {"P": p_count, "I": i_count, "O": o_count}
    best_label, best_count = max(counts.items(), key=lambda x: x[1])

    if best_count == 0:
        return "NONE"
    return best_label

In [ ]:
def build_sentence_dataset(doc_ids, docs, p_bio, i_bio, o_bio):#Build sentence-level training and test data
    rows = []

    for doc_id, tokens, p_tags, i_tags, o_tags in tqdm(zip(doc_ids, docs, p_bio, i_bio, o_bio), total=len(doc_ids)):
        if not (len(tokens) == len(p_tags) == len(i_tags) == len(o_tags)):
            continue

        sents, spans = split_tokens_into_sentences(tokens)

        for sent_idx, ((start, end), sent_tokens) in enumerate(zip(spans, sents)):
            p_sent = p_tags[start:end]
            i_sent = i_tags[start:end]
            o_sent = o_tags[start:end]

            sent_label = get_sentence_label(p_sent, i_sent, o_sent)
            sent_text = " ".join(sent_tokens)

            rows.append({
                "doc_id": doc_id,
                "sent_idx": sent_idx,
                "start": start,
                "end": end,
                "sentence": sent_text,
                "tokens": sent_tokens,
                "label": sent_label,
                "p_has": sentence_has_entity(p_sent),
                "i_has": sentence_has_entity(i_sent),
                "o_has": sentence_has_entity(o_sent),
            })

    return pd.DataFrame(rows)


train_sent_df = build_sentence_dataset(train_ids_final, train_docs_final, p_train_bio, i_train_bio, o_train_bio)
test_sent_df = build_sentence_dataset(test_ids_final, test_docs_final, p_test_bio, i_test_bio, o_test_bio)

print(train_sent_df.shape, test_sent_df.shape)
train_sent_df.head()

100%|██████████| 184/184 [00:00<00:00, 2605.63it/s]

(48257, 10) (2022, 10)


,doc_id,sent_idx,start,end,sentence,tokens,label,p_has,i_has,o_has
0,10036953,0,0,15,[ Triple therapy regimens involving H2 blockad...,"[[, Triple, therapy, regimens, involving, H2, ...",O,False,True,True
1,10036953,1,15,30,Comparison of ranitidine and lansoprazole in s...,"[Comparison, of, ranitidine, and, lansoprazole...",I,False,True,True
2,10036953,2,30,83,To evaluate the efficacy and safety of two 1-w...,"[To, evaluate, the, efficacy, and, safety, of,...",I,True,True,True
3,10036953,3,83,110,The drug combination and administration period...,"[The, drug, combination, and, administration, ...",I,False,True,False
4,10036953,4,110,130,"The ranitidine group received RNT 300 mg , CAM...","[The, ranitidine, group, received, RNT, 300, m...",I,False,True,False


In [19]:
train_sent_df = build_sentence_dataset(train_ids_final, train_docs_final, p_train_bio, i_train_bio, o_train_bio)
test_sent_df = build_sentence_dataset(test_ids_final, test_docs_final, p_test_bio, i_test_bio, o_test_bio)

print(train_sent_df.shape)
print(test_sent_df.shape)

train_sent_df.head()

100%|██████████| 184/184 [00:00<00:00, 2458.05it/s]

(48257, 10)
(2022, 10)


,doc_id,sent_idx,start,end,sentence,tokens,label,p_has,i_has,o_has
0,10036953,0,0,15,[ Triple therapy regimens involving H2 blockad...,"[[, Triple, therapy, regimens, involving, H2, ...",O,False,True,True
1,10036953,1,15,30,Comparison of ranitidine and lansoprazole in s...,"[Comparison, of, ranitidine, and, lansoprazole...",I,False,True,True
2,10036953,2,30,83,To evaluate the efficacy and safety of two 1-w...,"[To, evaluate, the, efficacy, and, safety, of,...",I,True,True,True
3,10036953,3,83,110,The drug combination and administration period...,"[The, drug, combination, and, administration, ...",I,False,True,False
4,10036953,4,110,130,"The ranitidine group received RNT 300 mg , CAM...","[The, ranitidine, group, received, RNT, 300, m...",I,False,True,False


In [ ]:
#Check whether the sentence segmentation is correct
sample_tokens = train_docs_final[0]
sample_doc_id = train_ids_final[0]

sents, spans = split_tokens_into_sentences(sample_tokens)

print("doc_id:", sample_doc_id)
print("Number of sentences:", len(sents))
print()

for i, (sent, span) in enumerate(zip(sents[:5], spans[:5])):
    print(f"sentence {i+1}:")
    print("text:", " ".join(sent))
    print("span:", span)
    print()

doc_id: 10036953
Number of sentences: 6

sentence 1:
text: [ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] .
span: (0, 15)

sentence 2:
text: Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection .
span: (15, 30)

sentence 3:
text: To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarithromycin ( CAM ) and metrinidazole ( MTZ ) .
span: (30, 83)

sentence 4:
text: The drug combination and administration periods in the PPI group were LPZ 30 mg , CAM 400 mg , MTZ 500 mg ( LCM group ) .
span: (83, 110)

sentence 5:
text: The ranitidine group received RNT 300 mg , CAM 400 mg , MTZ 500 mg ( RCM group ) .
span: (110, 130)



In [21]:
sample_idx = 0

print("doc_id:", train_ids_final[sample_idx])
print("Number of tokens:", len(train_docs_final[sample_idx]))
print("Number of P labels:", len(p_train_bio[sample_idx]))
print("Number of I labels:", len(i_train_bio[sample_idx]))
print("Number of O labels:", len(o_train_bio[sample_idx]))

doc_id: 10036953
Number of tokens: 162
Number of P labels: 162
Number of I labels: 162
Number of O labels: 162


In [23]:
print("Training set sentence label distribution:")
print(train_sent_df["label"].value_counts())
print()

print("Test set sentence label distribution:")
print(test_sent_df["label"].value_counts())

Training set sentence label distribution:
label
O       16801
NONE    12526
I       11872
P        7058
Name: count, dtype: int64

Test set sentence label distribution:
label
O       655
NONE    491
I       457
P       419
Name: count, dtype: int64


In [24]:
!pip install sentence-transformers scikit-learn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [25]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")#Convert each sentence into a numerical vector. 

train_sent_texts = train_sent_df["sentence"].tolist()
test_sent_texts = test_sent_df["sentence"].tolist()

X_train = embedder.encode(train_sent_texts, show_progress_bar=True)
X_test = embedder.encode(test_sent_texts, show_progress_bar=True)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1509 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

X_train shape: (48257, 384)
X_test shape: (2022, 384)


In [27]:
train_entity_mask = train_sent_df["label"] != "NONE"
cluster_df = train_sent_df[train_entity_mask].copy()
X_cluster = X_train[train_entity_mask.values]

label_to_int = {"P": 0, "I": 1, "O": 2}
y_cluster_int = np.array([label_to_int[x] for x in cluster_df["label"].tolist()])

print("Number of sentences used for clustering:", len(cluster_df))
print(cluster_df["label"].value_counts())

Number of sentences used for clustering: 35731
label
O    16801
I    11872
P     7058
Name: count, dtype: int64


In [28]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)#
kmeans_pred = kmeans.fit_predict(X_cluster)

print("=== KMeans ===")
print("ARI:", adjusted_rand_score(y_cluster_int, kmeans_pred))
print("NMI:", normalized_mutual_info_score(y_cluster_int, kmeans_pred))
print("Silhouette:", silhouette_score(X_cluster, kmeans_pred))

cluster_df["kmeans_cluster"] = kmeans_pred
print(pd.crosstab(cluster_df["label"], cluster_df["kmeans_cluster"]))

Exception in thread Thread-11 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\lenovo\anaconda3\envs\text_analytics\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\lenovo\anaconda3\envs\text_analytics\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\lenovo\anaconda3\envs\text_analytics\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xce in position 4: invalid continuation byte
c:\Users\lenovo\anaconda3\envs\text_analytics\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes

=== KMeans ===
ARI: 0.023014301851309926
NMI: 0.014629161920347088
Silhouette: 0.02642023004591465
kmeans_cluster     0     1     2
label                           
I               2387  3613  5872
O               3057  7847  5897
P               1957  2671  2430


In [29]:
hac = AgglomerativeClustering(n_clusters=3)
hac_pred = hac.fit_predict(X_cluster)

print("=== HAC ===")
print("ARI:", adjusted_rand_score(y_cluster_int, hac_pred))
print("NMI:", normalized_mutual_info_score(y_cluster_int, hac_pred))
print("Silhouette:", silhouette_score(X_cluster, hac_pred))

cluster_df["hac_cluster"] = hac_pred
print(pd.crosstab(cluster_df["label"], cluster_df["hac_cluster"]))

=== HAC ===
ARI: 0.003274818724501149
NMI: 0.009405287947113863
Silhouette: 0.009707450866699219
hac_cluster     0     1     2
label                        
I            8248   831  2793
O            9819  1118  5864
P            4461   811  1786


In [30]:
def map_cluster_to_majority_label(labels, cluster_ids):
    mapping = {}
    temp = pd.DataFrame({"label": labels, "cluster": cluster_ids})
    for c in sorted(temp["cluster"].unique()):
        majority = temp[temp["cluster"] == c]["label"].value_counts().idxmax()
        mapping[c] = majority
    return mapping

kmeans_map = map_cluster_to_majority_label(cluster_df["label"], cluster_df["kmeans_cluster"])
hac_map = map_cluster_to_majority_label(cluster_df["label"], cluster_df["hac_cluster"])

print("KMeans cluster -> label:", kmeans_map)
print("HAC cluster -> label:", hac_map)

KMeans cluster -> label: {np.int32(0): 'O', np.int32(1): 'O', np.int32(2): 'O'}
HAC cluster -> label: {np.int64(0): 'O', np.int64(1): 'O', np.int64(2): 'O'}
